In [ ]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    return -par[2]**2 / ((p - par[0]) - par[2])**2

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5]
q = df[:, 6]

err_p = df[:, 8]

err_p = df[:, 8]
err_q = df[:, 7]

p = np.delete(p, [0] )
q = np.delete(q, [0]  )
err_p = np.delete(err_p, [0] )
err_q = np.delete(err_q, [0] )


#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

Fit status: 0
Parametri stimati:
A = -7.070909528225954 ± 0.20682496600508218
B = -4.395189223321173 ± 0.15045082671131083
C = 200.28513815941233 ± 0.09131720811539819
Chi2 = 18277.777678208317
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      18277.8
NDf                       =          112
Edm                       =  2.51848e-08
NCalls                    =          137
A                         =     -7.07091   +/-   0.206825    
B                         =     -4.39519   +/-   0.150451    
C                         =      200.285   +/-   0.0913172   


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


In [26]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A NON è compatibile con 0
---
B NON è compatibile con 0


In [27]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff


#residuals_norm = np.delete(residuals_norm, [0,1, 10, 12, 14, 43, 44, 52])

#print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")


print()
print()

for i in np.arange(0,len(residuals_norm)):
    if residuals_norm[i] < -0.8:
        print (residuals_norm[i])
        print(i)

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


Fit status: 0
Parametri stimati:
const = 60.12859535894239 ± 8.011622362311794
#mu = 2.9523204305160196 ± 0.2523270302080072
#sigma = 2.154556010713915 ± 0.16599297222653486
Chi2 = 34.80247688937247


-13.806822472470197
61
-7.033630927397983
68
-2.848913430186412
69
-5.542773484263355
71
-12.950232968927637
72
-22.862715430098216
73
-29.61322473849864
74
-50.17906248530592
75
-61.525427023372195
76
-13.41256296626743
77
-58.98218266320692
87
-7.93999106436481
93
-0.8138325580909651
98
-9.362013611070836
100
-3.376720527274195
104
-7.733600523202225
105
-3.7353026871946153
106
-10.661097117568664
108
-12.174715318966994
109
-6.654893530284807
110
-7.34067254362101
112
-1.7113550826430222
113
-7.936405018002508
114
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      34.8025
NDf                       =           10
Edm                       =  3.74322e-06
NCalls                    =          181
const                     =      60.1286 

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [28]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [29]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[ 1.14503465e+02  1.41740603e+02  1.08580322e+02  1.08140764e+02
  1.03844986e+02  1.01482491e+02  9.21668649e+01  9.35977687e+01
  8.20814791e+01  7.79031435e+01  8.68684521e+01  6.92114580e+01
  7.13682483e+01  6.34976746e+01  7.47980987e+01  7.30178175e+01
  6.92872569e+01  6.34911402e+01  6.26850424e+01  5.66790691e+01
  5.90561843e+01  5.52762652e+01  5.77539532e+01  5.17011723e+01
  4.86298072e+01  4.47294211e+01  4.17943331e+01  4.50670421e+01
  4.48902310e+01  4.09379871e+01  4.46361332e+01  4.04948064e+01
  3.89869280e+01  3.68472375e+01  3.55613256e+01  3.37770825e+01
  3.36531786e+01  3.16721531e+01  3.25107187e+01  2.73439771e+01
  2.89823784e+01  2.74868268e+01  2.19208596e+01  1.96140430e+01
  2.41944593e+01  2.11612228e+01  2.20092543e+01  2.37021922e+01
  2.12952770e+01  2.12713047e+01  2.31291752e+01  1.89265699e+01
  2.10635407e+01  1.96386533e+01  2.11036326e+01  1.97434456e+01
  1.95606892e+01  1.60268189e+01  1.56657579e+01  1.51383249e+01
  1.14835803e+00 -4.63061

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created


In [30]:
# residui normalizzati divisi perzona di p

limiti = [232, 260, 316, 340, 460, 700, 2000]

print("zona di p      N   media   errore sulla media   std")
for i in range(len(limiti) - 1):
    sel = (p >= limiti[i]) & (p < limiti[i+1]) 
    r = residuals_norm[sel]
    print(limiti[i], "-", limiti[i+1], "  ", len(r), "  ", round(np.mean(r), 2), "  ",
          round(np.std(r) / np.sqrt(len(r)), 2), "  ", round(np.std(r), 2))

r = residuals_norm[:8]
print("primi 8 punti:", len(r), "  ", round(np.mean(r), 2), "  ",
      round(np.std(r) / np.sqrt(len(r)), 2), "  ", round(np.std(r), 2))

#  punti con residuo oltre 2 sigma
print("punti con |residuo| > 2:")
for i in range(len(p)):
    if abs(residuals_norm[i]) > 2:
        print("  indice", i, " p =", p[i], " residuo normalizzato =", round(residuals_norm[i], 1),
              " residuo [mm] =", round(q[i] - q_attesi[i], 1))



zona di p      N   media   errore sulla media   std
232 - 260    27    3.63    0.09    0.47
260 - 316    29    3.66    0.64    3.44
316 - 340    5    8.53    0.87    1.94
340 - 460    20    -3.98    4.94    22.1
460 - 700    17    4.68    4.47    18.45
700 - 2000    11    -5.39    1.27    4.22
primi 8 punti: 8    2.89    0.11    0.31
punti con |residuo| > 2:
  indice 0  p = 226.5  residuo normalizzato = 3.4  residuo [mm] = 114.5
  indice 1  p = 227.5  residuo normalizzato = 3.0  residuo [mm] = 141.7
  indice 2  p = 228.5  residuo normalizzato = 2.5  residuo [mm] = 108.6
  indice 3  p = 229.5  residuo normalizzato = 2.7  residuo [mm] = 108.1
  indice 4  p = 230.5  residuo normalizzato = 2.6  residuo [mm] = 103.8
  indice 5  p = 231.5  residuo normalizzato = 2.6  residuo [mm] = 101.5
  indice 6  p = 232.5  residuo normalizzato = 3.1  residuo [mm] = 92.2
  indice 7  p = 233.5  residuo normalizzato = 3.3  residuo [mm] = 93.6
  indice 8  p = 234.5  residuo normalizzato = 2.7  residuo [mm] =